## **Our next phase: Gold layer**

In [ ]:
from pyspark.sql.functions import *

In [2]:
# Read the Silver orders table
silver_orders = spark.read.table("silver.orders")

# Preview the order data
display(silver_orders.limit(5))

# Check the available columns
print(silver_orders.columns)

StatementMeta(, 995e32ab-5b27-4cd4-8936-8fe9890f45af, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 307ad71b-65b8-4fc7-afe3-fbb9fbda6a64)

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


## Create the Gold fact table

In [3]:
# Read the Silver orders table
silver_orders = spark.read.table("silver.orders")

# Create the Gold fact_orders table
gold_fact_orders = silver_orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).withColumn(
    "purchase_date",
    to_date(col("order_purchase_timestamp"))
).withColumn(
    "delivery_days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp")
    )
).withColumn(
    "delivery_status",
    when(
        col("order_status") != "Delivered",
        "Not Delivered"
    ).when(
        col("order_delivered_customer_date").isNull(),
        "Missing Delivery Date"
    ).when(
        col("order_delivered_customer_date") >
        col("order_estimated_delivery_date"),
        "Delivered Late"
    ).otherwise(
        "Delivered On Time"
    )
)

# Save the Gold table
gold_fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fact_orders")

# Verify the saved table
print("Gold fact_orders rows:", spark.read.table("gold.fact_orders").count())

display(spark.read.table("gold.fact_orders").limit(5))

StatementMeta(, 995e32ab-5b27-4cd4-8936-8fe9890f45af, 5, Finished, Available, Finished, False)

Gold fact_orders rows: 99441


SynapseWidget(Synapse.DataFrame, 218e6fe9-8839-4628-8ef1-ce2389e5be28)

## Create gold.fact_sales

In [5]:
# Read the Silver order items table
silver_order_items = spark.read.table("silver.order_items")

# Create the Gold sales fact table
gold_fact_sales = silver_order_items.select(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value"
).withColumn(
    "total_item_cost",
    col("price") + col("freight_value")
)

# Save the Gold table
gold_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fact_sales")

# Verify the saved table
print("Gold fact_sales rows:", spark.read.table("gold.fact_sales").count())

display(spark.read.table("gold.fact_sales").limit(5))

StatementMeta(, 995e32ab-5b27-4cd4-8936-8fe9890f45af, 7, Finished, Available, Finished, False)

Gold fact_sales rows: 112650


SynapseWidget(Synapse.DataFrame, 2f4d4a5e-1a4a-4bc2-bbe4-995c4fe91164)

## Create gold.dim_products

In [7]:
# Read the original product data
silver_products = spark.read.table("silver.products")

# Read the translation table
category_translation = spark.read.table("silver.category_translation")

# Standardize category names in both tables
products_clean = silver_products.withColumn(
    "category_key",
    lower(trim(regexp_replace(col("product_category_name"), " ", "_")))
)

translation_clean = category_translation.withColumn(
    "category_key",
    lower(trim(regexp_replace(col("product_category_name"), " ", "_")))
)


StatementMeta(, 3b79ab76-0d3b-4a23-a2f8-56f046e2897f, 9, Finished, Available, Finished, False)

In [10]:
# Join products to their English category names
gold_dim_products = silver_products.join(
    category_translation,
    on="product_category_name",
    how="left"
).select(
    col("product_id"),
    col("product_category_name").alias("product_category_name_portuguese"),
    col("product_category_name_english"),
    col("product_name_length"),
    col("product_description_length"),
    col("product_photos_quantity"),
    col("product_weight_g"),
    col("product_length_cm"),
    col("product_height_cm"),
    col("product_width_cm")
)

# Save the Gold product dimension
gold_dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_products")

# Verify the result
print("Gold product rows:", spark.read.table("gold.dim_products").count())

display(spark.read.table("gold.dim_products").limit(5))

StatementMeta(, 3b79ab76-0d3b-4a23-a2f8-56f046e2897f, 12, Finished, Available, Finished, False)

Gold product rows: 32951


SynapseWidget(Synapse.DataFrame, d4bbe776-267d-4851-8bb6-4f7d56dc1518)

## Create gold.dim_customers

In [9]:
# Read the Silver customers table
silver_customers = spark.read.table("silver.customers")

# Select the customer attributes for the Gold dimension
gold_dim_customers = silver_customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
)

# Save the Gold customer dimension
gold_dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_customers")

# Verify the saved table
print("Gold customer rows:", spark.read.table("gold.dim_customers").count())

display(spark.read.table("gold.dim_customers").limit(5))

StatementMeta(, 995e32ab-5b27-4cd4-8936-8fe9890f45af, 11, Finished, Available, Finished, False)

Gold customer rows: 99441


SynapseWidget(Synapse.DataFrame, 4717a2a3-d1e9-433e-934d-3bdf97062b7a)

## Create gold.dim_sellers

In [1]:
# Read the Silver sellers table
silver_sellers = spark.read.table("silver.sellers")

# Select the seller attributes for the Gold dimension
gold_dim_sellers = silver_sellers.select(
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
)

# Save the Gold seller dimension
gold_dim_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_sellers")

# Verify the saved table
print("Gold seller rows:", spark.read.table("gold.dim_sellers").count())

display(spark.read.table("gold.dim_sellers").limit(5))

StatementMeta(, cc33a8e5-57ff-4062-942b-db28adf2a58e, 3, Finished, Available, Finished, False)

Gold seller rows: 3095


SynapseWidget(Synapse.DataFrame, fbc0bade-894b-411d-8fd1-ddc17fece7f0)

## Create gold.dim_date

In [3]:
from pyspark.sql.functions import *

# Read the purchase timestamps from Silver
silver_orders = spark.read.table("silver.orders")

# Find the earliest and latest purchase dates
date_range = silver_orders.select(
    min(to_date(col("order_purchase_timestamp"))).alias("min_date"),
    max(to_date(col("order_purchase_timestamp"))).alias("max_date")
).first()

# Generate every date in that range
gold_dim_date = spark.sql(
    f"""
    SELECT explode(
        sequence(
            to_date('{date_range["min_date"]}'),
            to_date('{date_range["max_date"]}'),
            interval 1 day
        )
    ) AS date
    """
)

# Add useful calendar columns
gold_dim_date = gold_dim_date \
    .withColumn("year", year(col("date"))) \
    .withColumn("quarter", quarter(col("date"))) \
    .withColumn("month_number", month(col("date"))) \
    .withColumn("month_name", date_format(col("date"), "MMMM")) \
    .withColumn("day_of_month", dayofmonth(col("date"))) \
    .withColumn("day_of_week_number", dayofweek(col("date"))) \
    .withColumn("day_name", date_format(col("date"), "EEEE"))

# Save the Gold date dimension
gold_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_date")

# Verify the saved table
print("Gold date rows:", spark.read.table("gold.dim_date").count())

display(spark.read.table("gold.dim_date").limit(10))

StatementMeta(, cc33a8e5-57ff-4062-942b-db28adf2a58e, 5, Finished, Available, Finished, False)

Gold date rows: 774


SynapseWidget(Synapse.DataFrame, 73fa5c83-fb7b-4883-8ff0-aa99c0719cd2)

## Validation

In [4]:
# List the Gold tables created so far
gold_tables = [
    "fact_orders",
    "fact_sales",
    "dim_products",
    "dim_customers",
    "dim_sellers",
    "dim_date"
]

# Check each table
for table_name in gold_tables:
    table_df = spark.read.table(f"gold.{table_name}")
    print(f"{table_name}: {table_df.count()} rows")

StatementMeta(, cc33a8e5-57ff-4062-942b-db28adf2a58e, 6, Finished, Available, Finished, False)

fact_orders: 99441 rows
fact_sales: 112650 rows
dim_products: 32951 rows
dim_customers: 99441 rows
dim_sellers: 3095 rows
dim_date: 774 rows
